In [ ]:
pip install numpy pandas librosa matplotlib seaborn tqdm


In [ ]:
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from dataclasses import dataclass

# ===============================
# Data structure
# ===============================

@dataclass
class WorSegment:
    speaker: str
    text: str
    words: list  # [(word, start_ms, end_ms)]
    audio_path: Path
    cha_path: Path


# ===============================
# Utilities
# ===============================

def load_audio_duration(audio_path):
    y, sr = librosa.load(audio_path, sr=None)
    return len(y) / sr


def compute_speech_ratio(audio_path, segments):
    y, sr = librosa.load(audio_path, sr=None)
    energy = librosa.feature.rms(y=y)[0]
    threshold = np.percentile(energy, 75)

    speech_frames = energy > threshold
    speech_ratio = speech_frames.sum() / len(speech_frames)
    return speech_ratio


# ===============================
# Feature extraction
# ===============================

def extract_features(segment: WorSegment):
    words = segment.words

    starts = np.array([s for _, s, _ in words])
    ends = np.array([e for _, _, e in words])

    durations = ends - starts
    segment_duration = ends[-1] - starts[0]

    overlaps = np.sum(starts[1:] < ends[:-1])
    gaps = np.sum(starts[1:] - ends[:-1] > 500)

    return {
        "n_words": len(words),
        "segment_duration_ms": segment_duration,
        "avg_word_duration_ms": durations.mean(),
        "std_word_duration_ms": durations.std(),
        "words_per_sec": len(words) / (segment_duration / 1000),
        "overlap_count": overlaps,
        "gap_count": gaps
    }


# ===============================
# Main analysis pipeline
# ===============================

def analyze_dataset(segments):
    rows = []

    for seg in tqdm(segments, desc="Analyzing segments"):
        audio_duration = load_audio_duration(seg.audio_path)

        feats = extract_features(seg)
        speech_ratio = compute_speech_ratio(seg.audio_path, [seg])

        feats.update({
            "audio_duration_s": audio_duration,
            "speech_ratio": speech_ratio,
            "speaker": seg.speaker
        })

        rows.append(feats)

    df = pd.DataFrame(rows)
    return df


# ===============================
# Quality scoring (no Whisper)
# ===============================

def compute_quality_score(df):
    score = np.ones(len(df))

    score -= (df["overlap_count"] > 0) * 0.3
    score -= (df["gap_count"] > 2) * 0.2
    score -= (df["avg_word_duration_ms"] < 80) * 0.2
    score -= (df["avg_word_duration_ms"] > 1200) * 0.2
    score -= (df["speech_ratio"] < 0.3) * 0.3
    score -= (df["words_per_sec"] > 5) * 0.2

    return np.clip(score, 0, 1)


# ===============================
# Visualization
# ===============================

def plot_distributions(df):
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))

    sns.histplot(df["segment_duration_ms"], ax=axes[0, 0])
    axes[0, 0].set_title("Segment duration (ms)")

    sns.histplot(df["avg_word_duration_ms"], ax=axes[0, 1])
    axes[0, 1].set_title("Avg word duration (ms)")

    sns.histplot(df["words_per_sec"], ax=axes[0, 2])
    axes[0, 2].set_title("Words per second")

    sns.histplot(df["speech_ratio"], ax=axes[1, 0])
    axes[1, 0].set_title("Speech ratio")

    sns.histplot(df["overlap_count"], ax=axes[1, 1])
    axes[1, 1].set_title("Overlaps")

    sns.histplot(df["gap_count"], ax=axes[1, 2])
    axes[1, 2].set_title("Gaps")

    plt.tight_layout()
    plt.show()


def plot_correlation_matrix(df):
    plt.figure(figsize=(12, 8))
    corr = df.corr(numeric_only=True)
    sns.heatmap(corr, cmap="coolwarm", center=0)
    plt.title("Feature Correlation Matrix")
    plt.show()


def plot_quality_scatter(df):
    plt.figure(figsize=(8, 6))
    sns.scatterplot(
        data=df,
        x="words_per_sec",
        y="speech_ratio",
        hue="quality_score",
        palette="viridis"
    )
    plt.title("Words/sec vs Speech ratio (colored by quality)")
    plt.show()


# ===============================
# Example usage
# ===============================

if __name__ == "__main__":
    # 👉 À REMPLACER PAR TON PIPELINE
    segments = []  # liste de WorSegment déjà extraits

    df = analyze_dataset(segments)
    df["quality_score"] = compute_quality_score(df)

    print(df.describe())

    plot_distributions(df)
    plot_correlation_matrix(df)
    plot_quality_scatter(df)

    clean_df = df[df["quality_score"] > 0.7]
    print(f"\n✅ Clean segments kept: {len(clean_df)} / {len(df)}")
